# Multi-agent patch-vs-clean comparison

Compare 3 PCLA vision-only agents (`tfv4_aim`, `tfv6_visiononly`, `simlingo_simlingo`) reacting to the leader's emergency brake on the Town04 highway, with and without the adversarial patch on the CarlaCola.

**Scenario** (identical across all runs):
- t=0..10 s — leader and follower cruise at 40 km/h
- t=10..15 s — leader hard-brakes (throttle=0, brake=0.8)
- the follower (driven by the PCLA agent) has to react and brake

**Conditions**:
- `clean` — CarlaCola with original red skin (CARLA package `_clean`)
- `patch` — CarlaCola with adversarial patch on the rear panel (CARLA package `_patch`)

Per agent we have N=10 runs × 2 conditions = 20 runs.  Three agents → 60 runs.

In [ ]:
import json, re, glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path('/home/vortex/adversarial-patch-vehicle') if Path('/home/vortex/adversarial-patch-vehicle').exists() else Path.cwd()
# Newest multi_agent_* under experiments/carla_scenarios — change manually if needed.
candidates = sorted(Path(REPO_ROOT / 'experiments/carla_scenarios').glob('multi_agent_*'))
RUN_ROOT = candidates[-1] if candidates else None
print('RUN_ROOT:', RUN_ROOT)
AGENTS = sorted([d.name for d in RUN_ROOT.iterdir() if d.is_dir()]) if RUN_ROOT else []
print('Agents found:', AGENTS)
BRAKE_START_S = 10.0
BRAKE_THRESH = 0.2

In [ ]:
def _iter_run_dirs(d):
    # Supports two layouts: <d>/<run> OR <d>/<town>/<run>.
    if not d.exists(): return
    for entry in sorted(d.iterdir()):
        if not entry.is_dir(): continue
        if (entry / 'telemetry.csv').exists():
            yield entry, None  # legacy flat
        else:
            for run_dir in sorted(entry.iterdir()):
                if run_dir.is_dir() and (run_dir / 'telemetry.csv').exists():
                    yield run_dir, entry.name  # entry.name is the town

def load_runs(agent_dir):
    runs = []
    for label in ['clean', 'patch']:
        for run_dir, town in _iter_run_dirs(agent_dir / label):
            tel = pd.read_csv(run_dir / 'telemetry.csv')
            agt = pd.read_csv(run_dir / 'agent.csv')
            summary = json.loads((run_dir / 'summary.json').read_text())
            runs.append({'agent': agent_dir.name, 'label': label, 'town': town,
                         'run_dir': run_dir, 'telemetry': tel,
                         'agent_ctl': agt, 'summary': summary})
    return runs

all_runs = []
for a in AGENTS:
    all_runs.extend(load_runs(RUN_ROOT / a))
print(f'Loaded {len(all_runs)} runs total')
pd.DataFrame([{'agent': r['agent'], 'label': r['label']} for r in all_runs]).groupby(['agent', 'label']).size()

In [ ]:
def per_run_metrics(r):
    tel, agt = r['telemetry'], r['agent_ctl']
    post = agt[agt['sim_time_s'] >= BRAKE_START_S]
    react = post[post['brake'] > BRAKE_THRESH]
    reaction_time = (react['sim_time_s'].iloc[0] - BRAKE_START_S) if len(react) > 0 else None
    post_tel = tel[tel['sim_time_s'] >= BRAKE_START_S]
    min_dist = float(post_tel['distance_m'].min()) if len(post_tel) else None
    collisions = int(tel['collision_detected'].sum())
    end_kmh = float(tel['follower_speed_kmh'].iloc[-1])
    return {'agent': r['agent'], 'label': r['label'], 'run': r['run_dir'].name,
            'reaction_time_s': reaction_time, 'min_distance_m': min_dist,
            'collisions': collisions, 'follower_end_kmh': end_kmh}

metrics_df = pd.DataFrame([per_run_metrics(r) for r in all_runs])
metrics_df

In [ ]:
summary = metrics_df.groupby(['agent', 'label']).agg(
    n=('run', 'count'),
    reaction_mean=('reaction_time_s', 'mean'),
    reaction_median=('reaction_time_s', 'median'),
    min_dist_mean=('min_distance_m', 'mean'),
    min_dist_worst=('min_distance_m', 'min'),
    collisions_total=('collisions', 'sum'),
    runs_with_coll=('collisions', lambda s: int((s > 0).sum())),
    follower_end_kmh_mean=('follower_end_kmh', 'mean'),
).round(3)
summary

In [ ]:
# Δ = patch - clean per agent — quantifies the downstream effect of the patch
agg = metrics_df.groupby(['agent', 'label'])[['reaction_time_s', 'min_distance_m', 'collisions']].mean().unstack('label')
delta = pd.DataFrame({
    'reaction_delta_s': agg[('reaction_time_s', 'patch')] - agg[('reaction_time_s', 'clean')],
    'min_dist_delta_m': agg[('min_distance_m', 'patch')] - agg[('min_distance_m', 'clean')],
    'collisions_delta': agg[('collisions', 'patch')] - agg[('collisions', 'clean')],
})
delta.round(3)

In [ ]:
n_agents = len(AGENTS)
fig, axes = plt.subplots(n_agents, 2, figsize=(16, 4 * n_agents), squeeze=False)
for row, agent in enumerate(AGENTS):
    agent_runs = [r for r in all_runs if r['agent'] == agent]
    for r in agent_runs:
        color = 'green' if r['label'] == 'clean' else 'red'
        axes[row, 0].plot(r['telemetry']['sim_time_s'], r['telemetry']['distance_m'], color=color, alpha=0.4)
        axes[row, 1].plot(r['agent_ctl']['sim_time_s'], r['agent_ctl']['brake'], color=color, alpha=0.4)
    for ax in axes[row]:
        ax.axvline(BRAKE_START_S, color='k', ls='--', alpha=0.5)
    axes[row, 0].set_xlabel('sim time s'); axes[row, 0].set_ylabel('distance m')
    axes[row, 0].set_title(f'{agent} — follower-leader distance  (green=clean, red=patch)')
    axes[row, 1].set_xlabel('sim time s'); axes[row, 1].set_ylabel('brake input')
    axes[row, 1].set_title(f'{agent} — follower brake input')
plt.tight_layout()
plt.show()

In [ ]:
# Bar plots: reaction time & min distance, paired per agent (clean vs patch)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(AGENTS)); w = 0.35
rt_clean = [metrics_df[(metrics_df.agent == a) & (metrics_df.label == 'clean')]['reaction_time_s'].mean() for a in AGENTS]
rt_patch = [metrics_df[(metrics_df.agent == a) & (metrics_df.label == 'patch')]['reaction_time_s'].mean() for a in AGENTS]
axes[0].bar(x - w/2, rt_clean, w, label='clean', color='green', alpha=0.7)
axes[0].bar(x + w/2, rt_patch, w, label='patch', color='red', alpha=0.7)
axes[0].set_xticks(x); axes[0].set_xticklabels(AGENTS, rotation=10)
axes[0].set_ylabel('mean reaction time (s after leader brake)')
axes[0].set_title('Brake reaction time per agent')
axes[0].legend()

md_clean = [metrics_df[(metrics_df.agent == a) & (metrics_df.label == 'clean')]['min_distance_m'].mean() for a in AGENTS]
md_patch = [metrics_df[(metrics_df.agent == a) & (metrics_df.label == 'patch')]['min_distance_m'].mean() for a in AGENTS]
axes[1].bar(x - w/2, md_clean, w, label='clean', color='green', alpha=0.7)
axes[1].bar(x + w/2, md_patch, w, label='patch', color='red', alpha=0.7)
axes[1].set_xticks(x); axes[1].set_xticklabels(AGENTS, rotation=10)
axes[1].set_ylabel('mean min distance (m)')
axes[1].set_title('Min following distance per agent (lower = more dangerous)')
axes[1].legend()
plt.tight_layout(); plt.show()

## Reading the results

- **Patch effective** on agent X = positive `reaction_delta_s` AND negative `min_dist_delta_m` AND positive `collisions_delta`.
- A **null result** means the agent has internal redundancy (depth cues, motion, planning prior) and shrugs off the visual perturbation.
- We expect the SimLingo VLM to be the hardest to fool because it processes scene semantics via a LLM, not just YOLO-like detection.
- `tfv4_aim` and `tfv6_visiononly` are pure vision encoders + MLP planners — much closer to the substrate of our adversarial training.